# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets (by their @id)
print("Record Sets (cr:RecordSet) available in the dataset:")
record_sets = metadata.record_sets
if not record_sets:
    print('No record sets found in Croissant metadata; attempting to inspect files/distributions...')
    for dist in (metadata.distributions or []):
        print(f"Distribution: {getattr(dist, '@id', None)}")
else:
    for rs in record_sets:
        print(f"- @id: {rs['@id']} | name: {rs.get('name', '')}")

# Let's print Croissant metadata fields if any record set is present
selected_rs = None
if record_sets:
    selected_rs = record_sets[0]['@id']
    print(f"\nFields and columns for record set @id={selected_rs}:")
    for field in record_sets[0].get('fields', []):
        print(f"  -> Field @id: {field['@id']}  | Name: {field.get('name', '')}")
        for column in field.get('columns', []):
            print(f"      - Column @id: {column['@id']} | Name: {column.get('name', '')}")
else:
    print("\nNo record set structure present. You may need to extract by main data distribution instead.")

## 3. Data Extraction
Load data from a specific record set (or main distribution if no named record set) into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Try extracting all records from the available record set(s), else by default distribution
df_map = {}
# Try main Croissant logic first
try:
    if record_sets:
        # Use record set by @id
        for rs in record_sets:
            rsid = rs['@id']
            print(f"Extracting records from record set: {rsid}")
            records = list(dataset.records(record_set=rsid))
            df_map[rsid] = pd.DataFrame(records)
    else:
        # Try to guess the main record set from distributions
        print("No cr:RecordSet found. Using first distribution as main record set...")
        recs = list(dataset.records())
        # If records are dict-like
        if len(recs) > 0 and isinstance(recs[0], dict):
            df_map['main'] = pd.DataFrame(recs)
            print(f"Loaded DataFrame with columns: {df_map['main'].columns.tolist()}")
        else:
            print("No records could be loaded from dataset.")
except Exception as e:
    print(f"Extraction failure: {e}")
# Display columns of first DataFrame

main_df_id = list(df_map.keys())[0] if df_map else None
if main_df_id:
    print(f"\nFirst few columns of DataFrame ({main_df_id}):")
    print(df_map[main_df_id].columns.tolist())
    display(df_map[main_df_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section includes outlier removal, data transformation, and grouping by key attributes.

In [ ]:
# Pick a numeric field: typically protein, abundance, or coverage related field (e.g., 'MW_kDa' or 'Coverage')

import numpy as np

# List all numeric columns
candidate_numeric_cols = []
if main_df_id:
    for col in df_map[main_df_id].columns:
        if pd.api.types.is_numeric_dtype(df_map[main_df_id][col]):
            candidate_numeric_cols.append(col)

# If there are no detected numeric columns, try to coerce
if not candidate_numeric_cols:
    for col in df_map[main_df_id].columns:
        try:
            df_map[main_df_id][col] = pd.to_numeric(df_map[main_df_id][col], errors='coerce')
            if pd.api.types.is_numeric_dtype(df_map[main_df_id][col]):
                candidate_numeric_cols.append(col)
        except:
            continue

# Select a field for filtering (e.g., 'MW_kDa' or similar)
if candidate_numeric_cols:
    numeric_field = candidate_numeric_cols[0]
    print(f"Using numeric field for filtering and normalization: {numeric_field}")
    threshold = np.nanmean(df_map[main_df_id][numeric_field])  # Above mean

    filtered_df = df_map[main_df_id][df_map[main_df_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (above mean):")
    display(filtered_df.head())

    # Normalize that field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"First records of normalized {numeric_field}:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by a string/categorical field (e.g., first non-numeric column)
    candidate_group_fields = [c for c in df_map[main_df_id].columns if c not in candidate_numeric_cols]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f"Grouping data by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df.head())
else:
    print('No numeric field found for EDA analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df_id and candidate_numeric_cols:
    # Plot distribution of the main numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df_map[main_df_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group field available, plot mean per group
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        grouped.plot(kind='bar')
        plt.title(f'Mean {numeric_field} per {group_field} (filtered)')
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

else:
    print('No available numeric/group fields for visualization.')

## 6. Conclusion
This notebook demonstrated how to load, explore, and process the FAIR^2 Mass Spectrometry Analysis dataset using `mlcroissant` with Python. 
- We loaded the Croissant schema and inspected available record sets and columns via their `@id` fields.
- We extracted data into pandas DataFrames, filtered and normalized numeric fields, and demonstrated grouping and basic visualization.
- For more advanced analysis, further feature engineering or domain-specific preprocessing may be applied.

Feel free to adapt this notebook for downstream tasks such as machine learning modeling or custom visualizations!